This implementation has been picked up from the vide:

https://www.youtube.com/watch?v=VzS8hrOSSAs&list=PLTl9hO2Oobd97qfWC40gOSU8C0iu0m2l4&index=12

This Implementation covers English to Tamil

In [12]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [13]:
english_file = 'D:/Onedrive/experiments/experiments/AI/LLM/basics/dataset/samanantar/train.en/train.en'
tamil_file   = 'D:/Onedrive/experiments/experiments/AI/LLM/basics/dataset/samanantar/train.ta/train.ta'

START_TOKEN   = '<s>'
PADDING_TOKEN = '<pad>'
END_TOKEN     = '</s>'

tamil_letters = [
    'அ','ஆ','இ','ஈ','உ','ஊ','எ','ஏ','ஐ','ஒ','ஓ','ஔ','ஃ',
    'க','ங','ச','ஜ','ஞ','ட','ண','த','ந','ப','ம','ய','ர','ற','ல','ள','ழ','வ',
    'ஷ','ஸ','ஹ',  # Sanskrit‑origin letters occasionally present
]

# vowel signs / diacritics
vowel_signs = ['ா','ி','ீ','ு','ூ','ெ','ே','ை','ொ','ோ','ௌ','்','ௗ','ஂ']

# Tamil digits 0‑9 (U+0BE6‑U+0BEF) + ASCII digits 0‑9
ascii_digits  = [str(d) for d in range(10)]
tamil_digits  = ['௦','௧','௨','௩','௪','௫','௬','௭','௮','௯']

# ASCII punctuation identical to the original Kannada script list
ascii_punct = [' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',',
               '-', '.', '/', ':', '<', '=', '>', '?']

# Combine everything to form the Tamil vocabulary (order matters ‑ keep START
# first, PAD/END last to mirror original code)


tamil_vocabulary = (
    [START_TOKEN] +
    ascii_punct   +
    ascii_digits  +
    tamil_letters +
    vowel_signs   +
    tamil_digits  +
    [PADDING_TOKEN, END_TOKEN]
)

# -----------------------------------------------------------------------------
#  English vocabulary (unchanged from original)
# -----------------------------------------------------------------------------
english_vocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/',
                        '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
                        ':', '<', '=', '>', '?', '@',
                        'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L',
                        'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X',
                        'Y', 'Z',
                        '[', '\\', ']', '^', '_', '`',
                        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
                        'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x',
                        'y', 'z',
                        '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

In [14]:
index_to_tamil   = {k: v for k, v in enumerate(tamil_vocabulary)}
index_to_english = {k: v for k, v in enumerate(english_vocabulary)}

tamil_to_index   = {v: k for k, v in index_to_tamil.items()}
english_to_index = {v: k for k, v in index_to_english.items()}

with open(english_file, 'r', encoding='utf‑8') as f:
    english_sentences = [ln.rstrip('\n') for ln in f.readlines()]

with open(tamil_file, 'r', encoding='utf‑8') as f:
    tamil_sentences = [ln.rstrip('\n') for ln in f.readlines()]

TOTAL_SENTENCES = 100_000 # only this number of sentences are picked up to speed up computation.
english_sentences = english_sentences[:TOTAL_SENTENCES]
tamil_sentences   = tamil_sentences[:TOTAL_SENTENCES]

In [15]:
PERCENTILE = 97
print(f"97th percentile length Tamil  : {np.percentile([len(x) for x in tamil_sentences], PERCENTILE):.0f}")
print(f"97th percentile length English: {np.percentile([len(x) for x in english_sentences], PERCENTILE):.0f}")

max_sequence_length = 200 # used to discard sentences with over 200 tokens.

def is_valid_tokens(sentence, vocab):
    return all(tok in vocab for tok in set(sentence))


def is_valid_length(sentence, max_len):
    # leave room for END token
    return len(sentence) < (max_len - 1)

valid_idxs = []
for idx, (ta, en) in enumerate(zip(tamil_sentences, english_sentences)):
    if (is_valid_length(ta, max_sequence_length) and
        is_valid_length(en, max_sequence_length) and
        is_valid_tokens(ta, tamil_vocabulary)):
        valid_idxs.append(idx)

print(f"Total pairs loaded: {len(tamil_sentences)}")
print(f"Pairs kept (valid) : {len(valid_idxs)}")

tamil_sentences   = [tamil_sentences[i] for i in valid_idxs]
english_sentences = [english_sentences[i] for i in valid_idxs]

97th percentile length Tamil  : 238
97th percentile length English: 215
Total pairs loaded: 100000
Pairs kept (valid) : 23731


In [18]:
class TextDataset(Dataset): # for loading a custom dataset.
    def __init__(self, en_sents, ta_sents):
        self.en_sents = en_sents
        self.ta_sents = ta_sents

    def __len__(self):
        return len(self.en_sents)

    def __getitem__(self, idx): # getting corresponding English and Tamil sentence.
        return self.en_sents[idx], self.ta_sents[idx]


batch_size   = 3
dataset      = TextDataset(english_sentences, tamil_sentences)
train_loader = DataLoader(dataset, batch_size)

# Quick sanity print of first few batches
iterator = iter(train_loader)
for bnum, batch in enumerate(iterator):
    print(batch)
    if bnum > 3:
        break

[('The car has reportedly reached 100kmph from zero in under 3 seconds', 'Theres a lot of money involved.', '10,000 crores.'), ('இந்த கார் வெறும் 3 நொடிகளில் 100 கிமீ வேகத்தை எட்டும் ஆற்றல் கொண்டதாக இருக்கும்', 'அங்கு நிறைய பணம் வைக்கட்டிருக்கிறது.', '10,000 கோடிக்கு கையகப்படுத்தியது.')]
[('Theres never an opportunity to not work.', 'Open profile directory', 'Dont show info bar when pop-ups are blocked'), ('இதில் வேலை பறிபோக வாய்ப்பில்லை.', 'சுய விவரக்குறிப்பு கோப்பகத்தைத் திற', 'பாப்பப்கள் தடுக்கப்படும்போது தகவல் பட்டையை காட்டாதே')]
[('No specific person is mentioned in the notice.', 'Would anybody talk like that?', 'Australia cricket team'), ('எந்த குறிப்பிட்ட நபரும் பெயர் குறிப்பிடப்படவில்லை.', '"""""இதை யாரும் ஒண்ணும் சொல்ல மாட்டாங்களா?"', 'ஆஸ்திரேலிய கிரிக்கெட் அணி')]
[('Why must Gods servants keep exposing the man of lawlessness?', 'Mid-day meal scheme in schools', "We're taking the mark in the nest."), ('( பி) எந்தவிதத்தில் இது ஒரு புதிய கட்டளை?', 'பள்ளிகளில் காலை உணவு திட்டம்

In [19]:
def tokenize(sentence, lang_to_index, start_tok=True, end_tok=True):
    idxs = [lang_to_index[t] for t in sentence]
    if start_tok:
        idxs.insert(0, lang_to_index[START_TOKEN])
    if end_tok:
        idxs.append(lang_to_index[END_TOKEN])
    # pad
    idxs += [lang_to_index[PADDING_TOKEN]] * (max_sequence_length - len(idxs))
    return torch.tensor(idxs)

# Example tokenisation on first batch
eng_tok, ta_tok = [], []
for s in range(batch_size):
    en, ta = batch[0][s], batch[1][s]
    eng_tok.append(tokenize(en, english_to_index, start_tok=False, end_tok=False)) # for English since thats the I/P, we don't need to inject a start token.
    ta_tok.append(tokenize(ta, tamil_to_index,  start_tok=True,  end_tok=True)) # for Tamil, there's no starting Tamil word, hence we need a start token, also an end token is required.
eng_tok = torch.stack(eng_tok)
ta_tok  = torch.stack(ta_tok)
print(eng_tok.shape, ta_tok.shape)

torch.Size([3, 200]) torch.Size([3, 200])


In [ ]:
NEG_INFTY = -1e9 # to avoid NANs during loss computation, because if an entire row is 0 then softmax will screw it up.

def create_masks(eng_batch, ta_batch):
    n = len(eng_batch)
    look_ahead = torch.triu(torch.ones(max_sequence_length, max_sequence_length, dtype=torch.bool), diagonal=1)
    enc_pad   = torch.zeros(n, max_sequence_length, max_sequence_length, dtype=torch.bool) # A padding mask is added for the encoder.
    dec_pad_sa = torch.zeros_like(enc_pad)
    dec_pad_ca = torch.zeros_like(enc_pad)

    for i in range(n):
        eng_len = len(eng_batch[i])
        ta_len  = len(ta_batch[i])
        eng_mask_idx = np.arange(eng_len+1, max_sequence_length)
        ta_mask_idx  = np.arange(ta_len+1,  max_sequence_length)

        enc_pad[i, :, eng_mask_idx] = True
        enc_pad[i, eng_mask_idx, :] = True
        dec_pad_sa[i, :, ta_mask_idx] = True
        dec_pad_sa[i, ta_mask_idx, :] = True
        dec_pad_ca[i, :, eng_mask_idx] = True
        dec_pad_ca[i, ta_mask_idx, :]  = True

    enc_sa_mask = torch.where(enc_pad, NEG_INFTY, 0)
    dec_sa_mask = torch.where(look_ahead | dec_pad_sa, NEG_INFTY, 0)
    dec_ca_mask = torch.where(dec_pad_ca, NEG_INFTY, 0)
    return enc_sa_mask, dec_sa_mask, dec_ca_mask

# run once to verify dims
create_masks(batch[0], batch[1])

(tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          ...,
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          ...,
    

In [9]:
class SentenceEmbedding(nn.Module):
    """Create an embedding for a sentence."""
    def __init__(self, max_len, d_model, lang_to_idx, start_tok, end_tok, pad_tok):
        super().__init__()
        self.vocab_size = len(lang_to_idx)
        self.max_len    = max_len
        self.embedding  = nn.Embedding(self.vocab_size, d_model)
        self.lang_to_idx = lang_to_idx
        self.position   = PositionalEncoding(d_model, max_len)
        self.drop       = nn.Dropout(0.1)
        self.start_tok  = start_tok
        self.end_tok    = end_tok
        self.pad_tok    = pad_tok

    def batch_tokenize(self, batch, add_start=True, add_end=True):
        def _tok(s):
            idxs = [self.lang_to_idx[ch] for ch in s]
            if add_start:
                idxs.insert(0, self.lang_to_idx[self.start_tok])
            if add_end:
                idxs.append(self.lang_to_idx[self.end_tok])
            idxs += [self.lang_to_idx[self.pad_tok]] * (self.max_len - len(idxs))
            return torch.tensor(idxs)
        return torch.stack([_tok(sent) for sent in batch])

    def forward(self, batch, add_end=True):
        x = self.batch_tokenize(batch, add_end=add_end).to(next(self.parameters()).device)
        x = self.embedding(x) + self.position().to(x.device)
        return self.drop(x)